# Handling Missing Values Using Drop and Fill Strategies

This notebook demonstrates intentional, well-reasoned data cleaning decisions using drop and fill strategies for handling missing values in Pandas DataFrames.

## Part 1: Loading and Inspecting Data with Missing Values

Before applying any handling strategy, we must first detect and understand the extent of missing data.

In [ ]:
import pandas as pd
import numpy as np

# Load datasets with missing values
employees_df = pd.read_csv('../employees_with_missing.csv')
sales_df = pd.read_csv('../sales_with_missing.csv')

print("="*70)
print("ORIGINAL EMPLOYEES DATAFRAME (WITH MISSING VALUES)")
print("="*70)
print(employees_df)
print(f"\nShape: {employees_df.shape}")  # (rows, columns)

print("\n" + "="*70)
print("ORIGINAL SALES DATAFRAME (WITH MISSING VALUES)")
print("="*70)
print(sales_df)
print(f"\nShape: {sales_df.shape}")

### 1.1: Initial Missing Value Assessment

Understand the extent of missing data before deciding on a strategy.

In [ ]:
print("MISSING VALUE SUMMARY - EMPLOYEES")
print("="*70)

# Count and percentage of missing values
missing_summary = pd.DataFrame({
    'Column': employees_df.columns,
    'Missing_Count': employees_df.isna().sum(),
    'Missing_Percentage': (employees_df.isna().sum() / len(employees_df) * 100).round(1),
    'Data_Type': employees_df.dtypes
})

print(missing_summary)

print("\n" + "="*70)
print("ROWS AFFECTED BY MISSING VALUES")
print("="*70)

# Show which rows have missing values
rows_with_missing = employees_df[employees_df.isna().any(axis=1)]
print(f"\nRows with at least one missing value: {len(rows_with_missing)} out of {len(employees_df)}")
print("\nAffected rows:")
print(rows_with_missing[["EmployeeID", "Name", "Department", "Salary", "HireDate", "PhoneNumber"]])

## Part 2: Drop Strategies

Dropping removes rows or columns with missing data. This is appropriate when:
- The column/row is not critical to the analysis
- Data is Missing Completely at Random (MCAR)
- The percentage of missing data is low
- You can afford to lose data

### 2.1: Drop Rows with ANY Missing Values

Using `.dropna()` without parameters removes any row with at least one missing value.

In [ ]:
print("STRATEGY 1: DROP ROWS WITH ANY MISSING VALUES")
print("="*70)

# Original state
print(f"Original shape: {employees_df.shape}")
print(f"Rows: {len(employees_df)}, Columns: {len(employees_df.columns)}")

# Apply dropna()
employees_drop_any = employees_df.dropna()

print(f"\nAfter dropna(): {employees_drop_any.shape}")
print(f"Rows: {len(employees_drop_any)}, Columns: {len(employees_drop_any.columns)}")

# Impact
rows_lost = len(employees_df) - len(employees_drop_any)
pct_lost = (rows_lost / len(employees_df)) * 100

print(f"\nIMPACT: {rows_lost} rows removed ({pct_lost:.1f}% data loss)")

print("\nRemaining employees:")
print(employees_drop_any[["EmployeeID", "Name", "Department", "Salary"]])

print("\n✓ When to use this strategy:")
print("  - Dataset is large and can afford data loss")
print("  - Missing data is scattered across many columns")
print("  - You want guaranteed complete rows")

### 2.2: Drop Rows Missing Specific Columns

Using `.dropna(subset=[...])` is more selective - only removes rows missing values in critical columns.

In [ ]:
print("STRATEGY 2: DROP ROWS MISSING SPECIFIC COLUMNS (Selective)")
print("="*70)

# Original state
print(f"Original shape: {employees_df.shape}")
print(f"Original rows: {len(employees_df)}")

# Drop rows missing critical columns: Salary and Department
critical_columns = ['Salary', 'Department']
employees_drop_critical = employees_df.dropna(subset=critical_columns)

print(f"\nAfter dropna(subset={critical_columns}): {employees_drop_critical.shape}")
print(f"Rows: {len(employees_drop_critical)}, Columns: {len(employees_drop_critical.columns)}")

# Impact
rows_lost = len(employees_df) - len(employees_drop_critical)
pct_lost = (rows_lost / len(employees_df)) * 100

print(f"\nIMPACT: {rows_lost} rows removed ({pct_lost:.1f}% data loss)")

print("\nRemaining employees:")
print(employees_drop_critical)

print("\n✓ When to use this strategy:")
print("  - Specific columns are critical for your analysis")
print("  - Missing data is concentrated in non-critical columns")
print("  - You want to preserve more data than dropna()")
print("  - Column importance matters more than row completeness")

### 2.3: Drop Columns with Missing Values

Using `.dropna(axis=1)` removes entire columns that contain any missing values.

In [ ]:
print("STRATEGY 3: DROP COLUMNS WITH MISSING VALUES")
print("="*70)

# Original state
print(f"Original shape: {employees_df.shape}")
print(f"Columns: {employees_df.columns.tolist()}")

# Drop columns with ANY missing values
employees_drop_cols = employees_df.dropna(axis=1)

print(f"\nAfter dropna(axis=1): {employees_drop_cols.shape}")
print(f"Rows: {len(employees_drop_cols)}, Columns: {len(employees_drop_cols.columns)}")
print(f"Remaining columns: {employees_drop_cols.columns.tolist()}")

# Impact
cols_lost = len(employees_df.columns) - len(employees_drop_cols.columns)
print(f"\nIMPACT: {cols_lost} columns removed")
dropped_cols = [col for col in employees_df.columns if col not in employees_drop_cols.columns]
print(f"Dropped columns: {dropped_cols}")

print("\nRemaining data:")
print(employees_drop_cols)

print("\n✓ When to use this strategy:")
print("  - Entire columns are unreliable or incomplete")
print("  - Missing columns are optional to your analysis")
print("  - You want to preserve all rows")
print("  - Data loss in features is acceptable")

### 2.4: Drop Rows with Threshold

Using `.dropna(thresh=n)` removes rows with fewer than `n` non-null values.

In [ ]:
print("STRATEGY 4: DROP ROWS WITH INSUFFICIENT DATA (Threshold)")
print("="*70)

# Original state
print(f"Original shape: {employees_df.shape}")
print(f"Total columns: {len(employees_df.columns)}")

# Keep rows with at least 4 non-null values (out of 6)
min_values_needed = 4
employees_drop_thresh = employees_df.dropna(thresh=min_values_needed)

print(f"\nAfter dropna(thresh={min_values_needed}):")
print(f"Rows: {len(employees_drop_thresh)}, Columns: {len(employees_drop_thresh.columns)}")
print(f"Shape: {employees_drop_thresh.shape}")

# Impact
rows_lost = len(employees_df) - len(employees_drop_thresh)
pct_lost = (rows_lost / len(employees_df)) * 100

print(f"\nIMPACT: {rows_lost} rows removed ({pct_lost:.1f}% data loss)")
print(f"(Kept only rows with {min_values_needed}+ non-null values)")

print("\nRemaining employees:")
print(employees_drop_thresh)

print("\n✓ When to use this strategy:")
print("  - You need a minimum amount of data per row")
print("  - Row completeness is partially important")
print("  - You want flexibility between strict and lenient approaches")

## Part 3: Fill Strategies

Filling replaces missing values with estimated or placeholder data. This is appropriate when:
- The column is critical and you need to preserve rows
- Missing values are not random (MNAR)
- You have a reasonable estimate for the missing value
- Data loss would be problematic

### 3.1: Fill with Constant Values

Using `.fillna(value)` replaces missing values with a specific constant.

In [ ]:
print("STRATEGY 1: FILL WITH CONSTANT VALUES")
print("="*70)

# Create a copy to demonstrate
employees_fill_const = employees_df.copy()

print(f"Original shape: {employees_fill_const.shape}")
print(f"Missing values: {employees_fill_const.isna().sum().sum()} total")

print("\nBefore filling:")
print(employees_fill_const[["EmployeeID", "Name", "Department", "PhoneNumber"]])

# Fill PhoneNumber with 'Not Provided' (non-critical field)
employees_fill_const['PhoneNumber'] = employees_fill_const['PhoneNumber'].fillna('Not Provided')

# Fill HireDate with 'Unknown Date'
employees_fill_const['HireDate'] = employees_fill_const['HireDate'].fillna('Unknown Date')

print("\nAfter filling with constants:")
print(employees_fill_const[["EmployeeID", "Name", "Department", "HireDate", "PhoneNumber"]])

print(f"\nShape after fill: {employees_fill_const.shape}")
print(f"Missing values remaining: {employees_fill_const.isna().sum().sum()} total")

print("\n✓ When to use this strategy:")
print("  - Missing values represent a specific category (e.g., 'Unknown')")
print("  - The column is categorical with missing = specific meaning")
print("  - You want to preserve all rows")
print("  - Constant doesn't alter analysis (truly unknown data)")

### 3.2: Fill with Statistical Values (Mean, Median)

For numeric columns, fill missing values with mean or median of existing data.

In [ ]:
print("STRATEGY 2: FILL WITH STATISTICAL VALUES")
print("="*70)

# Work with a numeric column: Salary
print("Column: Salary (numeric)")
print(f"\nOriginal Salary values:")
print(employees_df[["EmployeeID", "Name", "Salary"]])

print(f"\nMissing salary count: {employees_df['Salary'].isna().sum()}")

# Calculate statistics
salary_mean = employees_df['Salary'].mean()
salary_median = employees_df['Salary'].median()

print(f"\nSalary statistics (from non-missing values):")
print(f"  Mean: ${salary_mean:,.2f}")
print(f"  Median: ${salary_median:,.2f}")

# Fill with mean
employees_fill_mean = employees_df.copy()
employees_fill_mean['Salary'] = employees_fill_mean['Salary'].fillna(salary_mean)

print(f"\nAfter filling with mean:")
print(employees_fill_mean[["EmployeeID", "Name", "Salary"]])

# Fill with median
employees_fill_median = employees_df.copy()
employees_fill_median['Salary'] = employees_fill_median['Salary'].fillna(salary_median)

print(f"\nAfter filling with median:")
print(employees_fill_median[["EmployeeID", "Name", "Salary"]])

print("\n✓ When to use this strategy:")
print("  - Column is numeric (temperature, salary, age, etc.)")
print("  - Missing data is likely Missing at Random (MAR)")
print("  - You want to preserve rows while estimating values")
print("  - Sample statistics are reasonable estimates")

print("\n⚠ Mean vs Median:")
print("  - Use MEAN for normally distributed data")
print("  - Use MEDIAN for skewed data or when outliers exist")

### 3.3: Forward Fill and Backward Fill

For time-series or sequential data, propagate values forward or backward.

In [ ]:
print("STRATEGY 3: FORWARD & BACKWARD FILL (for Sequential Data)")
print("="*70)

# Use sales data which is time-ordered
print("Working with Sales Data (time-ordered):")
print("\nOriginal data:")
print(sales_df[["Date", "Product", "Quantity", "Total"]])

print(f"\nMissing Products: {sales_df['Product'].isna().sum()}")
print(f"Missing Quantities: {sales_df['Quantity'].isna().sum()}")

# Forward fill: use previous row's value
sales_ffill = sales_df.copy()
sales_ffill['Product'] = sales_ffill['Product'].fillna(method='ffill')
sales_ffill['Quantity'] = sales_ffill['Quantity'].fillna(method='ffill')

print("\nAfter Forward Fill (.fillna(method='ffill')):")
print(sales_ffill[["Date", "Product", "Quantity", "Total"]])

print("\n" + "="*70)

# Backward fill: use next row's value
sales_bfill = sales_df.copy()
sales_bfill['Product'] = sales_bfill['Product'].fillna(method='bfill')
sales_bfill['Quantity'] = sales_bfill['Quantity'].fillna(method='bfill')

print("\nAfter Backward Fill (.fillna(method='bfill')):")
print(sales_bfill[["Date", "Product", "Quantity", "Total"]])

print("\n✓ When to use these strategies:")
print("  - Data is time-series or sequential")
print("  - Values change gradually over time")
print("  - Forward fill: assume value persists until new value")
print("  - Backward fill: assume upcoming value applies to gap")

### 3.4: Fill by Group (Groupby-based Filling)

Fill missing values using statistics calculated within groups.

In [ ]:
print("STRATEGY 4: FILL BY GROUP (Context-aware Filling)")
print("="*70)

# Fill salary by department median
employees_fill_group = employees_df.copy()

print("\nOriginal data:")
print(employees_fill_group[["EmployeeID", "Name", "Department", "Salary"]])

print(f"\nMissing Salary values: {employees_df['Salary'].isna().sum()}")
print(f"Missing Department values: {employees_df['Department'].isna().sum()}")

# Calculate salary by department (ignoring missing)
print("\nSalary by Department (before filling):")
dept_stats = employees_df.groupby('Department')['Salary'].agg(['count', 'mean', 'median'])
print(dept_stats)

# Fill salary with department median (only where department is known)
# For rows where we know the department, use the department median
for dept in employees_fill_group['Department'].dropna().unique():
    dept_median = employees_df[employees_df['Department'] == dept]['Salary'].median()
    mask = (employees_fill_group['Department'] == dept) & (employees_fill_group['Salary'].isna())
    employees_fill_group.loc[mask, 'Salary'] = dept_median

print("\nAfter filling Salary with department-specific median:")
print(employees_fill_group[["EmployeeID", "Name", "Department", "Salary"]])

# For rows with unknown department, still have missing salary
print(f"\nMissing Salary after group fill: {employees_fill_group['Salary'].isna().sum()}")
print("(These employees have unknown department, so we can't use department median)")

print("\n✓ When to use this strategy:")
print("  - Missing values depend on categories (departments, regions, etc.)")
print("  - Different groups have different value distributions")
print("  - Context matters for estimation")
print("  - More sophisticated than global mean/median")

## Part 4: Comparison of Strategies

Let's compare how different strategies affect the dataset.

### 4.1: Impact Summary - Employees Dataset

In [ ]:
print("STRATEGY COMPARISON - IMPACT ON EMPLOYEES DATASET")
print("="*70)

# Original
original_shape = employees_df.shape
original_missing = employees_df.isna().sum().sum()

# Strategy 1: Drop all missing
drop_all = employees_df.dropna()

# Strategy 2: Drop critical columns only
drop_critical = employees_df.dropna(subset=['Salary', 'Department'])

# Strategy 3: Drop columns
drop_cols = employees_df.dropna(axis=1)

# Strategy 4: Fill (combined approach)
employees_filled = employees_df.copy()
employees_filled['PhoneNumber'] = employees_filled['PhoneNumber'].fillna('Not Provided')
employees_filled['HireDate'] = employees_filled['HireDate'].fillna('Unknown')
# Fill salary with mean (where department is known)
salary_mean = employees_df['Salary'].mean()
for dept in employees_filled['Department'].dropna().unique():
    dept_median = employees_df[employees_df['Department'] == dept]['Salary'].median()
    mask = (employees_filled['Department'] == dept) & (employees_filled['Salary'].isna())
    employees_filled.loc[mask, 'Salary'] = dept_median

# Create comparison table
comparison = pd.DataFrame({
    'Strategy': ['Original', 'Drop Any Missing', 'Drop Critical Cols', 'Drop Columns', 'Fill (Mixed)'],
    'Rows': [original_shape[0], len(drop_all), len(drop_critical), len(drop_cols), len(employees_filled)],
    'Columns': [original_shape[1], original_shape[1], original_shape[1], len(drop_cols.columns), original_shape[1]],
    'Total Missing': [original_missing, drop_all.isna().sum().sum(), drop_critical.isna().sum().sum(), 
                      drop_cols.isna().sum().sum(), employees_filled.isna().sum().sum()],
    'Data Loss %': [0, round((1 - len(drop_all)/original_shape[0])*100, 1), 
                    round((1 - len(drop_critical)/original_shape[0])*100, 1),
                    round((1 - len(drop_cols)/original_shape[0])*100, 1), 0]
})

print(comparison.to_string(index=False))

print("\n" + "="*70)
print("Key Observations:")
print("="*70)
print(f"✓ Drop Any: Loses {original_shape[0] - len(drop_all)} rows (strictest)")
print(f"✓ Drop Critical: Loses {original_shape[0] - len(drop_critical)} rows (moderate)")
print(f"✓ Drop Columns: Loses {original_shape[1] - len(drop_cols.columns)} columns (column-focused)")
print(f"✓ Fill: Preserves ALL rows (most data-preserving)")

### 4.2: Detailed Comparison - What Each Strategy Retains

In [ ]:
print("DETAILED COMPARISON: WHAT EACH STRATEGY KEEPS")
print("="*70)

employee_ids = [1001, 1002, 1003, 1004, 1005, 1006, 1008, 1009]

# Check which employees remain in each strategy
print("\nEmployee Retention Matrix:")
retention = pd.DataFrame({
    'EmployeeID': employee_ids,
    'Drop Any': [emp_id in drop_all['EmployeeID'].values for emp_id in employee_ids],
    'Drop Critical': [emp_id in drop_critical['EmployeeID'].values for emp_id in employee_ids],
    'Drop Cols': [emp_id in drop_cols['EmployeeID'].values for emp_id in employee_ids],
    'Fill (Mixed)': [emp_id in employees_filled['EmployeeID'].values for emp_id in employee_ids]
})

print(retention.to_string(index=False))

print("\nTrue = Employee retained, False = Employee removed")
print("\nSpecific examples:")
print(f"Employee 1002 (Bob): Missing Department")
print(f"  - Drop Any: Removed ✗ (has ANY missing)")
print(f"  - Drop Critical: Removed ✗ (Department is critical)")
print(f"  - Drop Cols: Retained ✓ (drops columns, not rows)")
print(f"  - Fill: Retained ✓ (can fill non-critical fields)")

print(f"\nEmployee 1006 (Unknown): Missing Name only")
print(f"  - Drop Any: Removed ✗")
print(f"  - Drop Critical: Retained ✓ (has Salary and Department)")
print(f"  - Drop Cols: Retained ✓")
print(f"  - Fill: Retained ✓")

## Part 5: Decision-Making Framework

How to choose between strategies based on your data and analysis needs.

In [ ]:
print("DECISION-MAKING FRAMEWORK FOR HANDLING MISSING VALUES")
print("="*70)

print("\n1. ASSESS THE MISSING DATA")
print("-" * 70)
print("   a) How much is missing?")
print("      - < 5%: Can afford to drop (usually safe)")
print("      - 5-20%: Consider selective dropping or filling")
print("      - > 20%: Strongly consider filling to preserve data")
print("   b) Which columns are missing?")
print("      - Critical columns: Usually best to fill or drop rows")
print("      - Optional columns: Safe to drop the column")
print("   c) Is data random or systematic?")
print("      - Random (MCAR): Either drop or fill works")
print("      - Not random (MAR/MNAR): Filling is usually safer")

print("\n2. CHOOSE STRATEGY BY COLUMN TYPE")
print("-" * 70)
print("   NUMERIC COLUMN (Salary, Price, etc.):")
print("   ✓ Drop: If < 5% missing AND row loss is acceptable")
print("   ✓ Fill Mean/Median: If > 5% missing AND value is critical")
print("   ✓ Fill by Group: If column varies by category")

print("\n   CATEGORICAL COLUMN (Department, Product, etc.):")
print("   ✓ Drop: If < 10% missing OR column is optional")
print("   ✓ Fill: With most common category (mode)")
print("   ✓ Fill by Group: If missingness patterns differ by subgroup")

print("\n   DATE COLUMN (HireDate, Timestamp, etc.):")
print("   ✓ Drop: If column is not critical to analysis")
print("   ✓ Fill: With 'Unknown' or placeholder")
print("   ✓ Forward/Backward fill: If time-series data")

print("\n3. CRITICAL QUESTIONS BEFORE DECIDING")
print("-" * 70)
print("   ❓ Can I afford to lose these rows?")
print("      YES → Consider dropping")
print("      NO → Consider filling")

print("\n   ❓ Is this column essential for my analysis?")
print("      YES → Keep the column (drop/fill as needed)")
print("      NO → Consider dropping the column")

print("\n   ❓ Do I have a reasonable estimate for missing values?")
print("      YES → Fill with the estimate")
print("      NO → Consider dropping")

print("\n   ❓ Will my analysis be sensitive to imputed values?")
print("      YES → Be conservative (consider dropping)")
print("      NO → Filling is acceptable")

print("\n4. QUICK REFERENCE DECISION TABLE")
print("-" * 70)
decision_table = pd.DataFrame({
    'Missing %': ['< 5%', '< 5%', '5-20%', '5-20%', '> 20%', '> 20%'],
    'Column Type': ['Numeric', 'Critical', 'Numeric', 'Critical', 'Numeric', 'Critical'],
    'Recommended': ['Drop rows', 'Drop rows', 'Fill mean/median', 'Fill by group', 'Fill by group', 'Fill (stratified)']
})
print(decision_table.to_string(index=False))

## Part 6: Scenario Analysis - Critical Column Challenge

Like the video scenario: A critical numeric column has missing values. How do we decide?

In [ ]:
print("SCENARIO: MISSING VALUES IN CRITICAL NUMERIC COLUMN")
print("="*70)

print("\nSituation: Salary column has 2 missing values (20% of data)")
print("Impact: Analysis needs salary for budgeting and compensation")

print("\n" + "="*70)
print("OPTION 1: DROP ROWS (Risky!)")
print("="*70)

# Show what happens if we drop
drop_option = employees_df.dropna(subset=['Salary'])

print(f"\nBefore: {len(employees_df)} employees")
print(f"After:  {len(drop_option)} employees")
print(f"\nLost employees: {len(employees_df) - len(drop_option)}")

print("\nEmployees removed:")
removed = employees_df[employees_df['Salary'].isna()][['EmployeeID', 'Name', 'Department']]
print(removed)

print("\n⚠ RISKS of dropping:")
print("  - Lost 2 full employee records")
print("  - One engineer (Charlie) - expensive to lose")
print("  - Reduced sample size affects metrics")
print("  - May bias remaining data (if missing wasn't random)")
print("  - Later discover important employees were excluded")

print("\n" + "="*70)
print("OPTION 2: FILL VALUES (Safer!)")
print("="*70)

# Show what happens if we fill
fill_option = employees_df.copy()

# Fill with department median
print("\nApproach A: Fill with Department Median")
for dept in fill_option['Department'].dropna().unique():
    dept_median = employees_df[employees_df['Department'] == dept]['Salary'].median()
    mask = (fill_option['Department'] == dept) & (fill_option['Salary'].isna())
    if mask.any():
        print(f"  - {dept} median: ${dept_median:,.0f}")
    fill_option.loc[mask, 'Salary'] = dept_median

# Show what was filled
filled_rows = employees_df[employees_df['Salary'].isna()].copy()
for idx in filled_rows.index:
    print(f"  - Employee {fill_option.loc[idx, 'EmployeeID']}: Filled with ${fill_option.loc[idx, 'Salary']:,.0f}")

print(f"\nBefore: {len(employees_df)} employees with {employees_df['Salary'].isna().sum()} missing")
print(f"After:  {len(fill_option)} employees with {fill_option['Salary'].isna().sum()} missing")

print("\n✓ BENEFITS of filling:")
print("  - Preserved all employee records")
print("  - Used context (department) for estimation")
print("  - Reasonable salary estimates based on peers")
print("  - Maintained sample size for analysis")
print("  - Can flag imputed values for later review")

print("\n⚠ TRADE-OFFS of filling:")
print("  - Introduced estimated values (not real data)")
print("  - May bias results if estimates are wrong")
print("  - Need to document which values were imputed")
print("  - Shouldn't use in contexts requiring actual values")

print("\n" + "="*70)
print("RECOMMENDATION")
print("="*70)
print("\nFor a CRITICAL numeric column with missing values:")
print("\n→ FILL is usually safer than DROP when:")
print("  • Column is essential to the analysis")
print("  • You have a reasonable estimation method")
print("  • Losing rows would compromise results")
print("  • Missing data is concentrated (< 30%)")

print("\n→ DROP is appropriate when:")
print("  • Missing data is truly random")
print("  • You have a large sample (can afford loss)")
print("  • You want to maintain data integrity (no imputation)")
print("  • Later you discover which rows were incomplete")

print("\n→ BEST PRACTICE:")
print("  1. Fill with reasonable estimates")
print("  2. Create a marker column (e.g., 'salary_imputed')")
print("  3. Document the imputation method")
print("  4. Run sensitivity analysis with and without imputed data")
print("  5. Decide if results are robust")

## Part 7: Implementation Checklist

A practical checklist for handling missing values in your own analysis.

In [ ]:
print("IMPLEMENTATION CHECKLIST")
print("="*70)

print("\n☐ STEP 1: DETECT & DOCUMENT")
print("  ☐ Load data and run .info() to identify missing values")
print("  ☐ Calculate percentage missing per column")
print("  ☐ Identify which rows are affected")
print("  ☐ Document findings in comments or report")

print("\n☐ STEP 2: ANALYZE PATTERNS")
print("  ☐ Check if missing data is random or systematic")
print("  ☐ Look for co-occurrences (columns that often miss together)")
print("  ☐ Consider why data might be missing")

print("\n☐ STEP 3: PLAN STRATEGY")
print("  ☐ For each column, decide: Keep, Drop, or Fill?")
print("  ☐ Document the reasoning for each decision")
print("  ☐ Consider impact on downstream analysis")

print("\n☐ STEP 4: IMPLEMENT & VERIFY")
print("  ☐ Apply dropna() or fillna() methods")
print("  ☐ Check new dataset shape and structure")
print("  ☐ Verify no unintended missing values remain")
print("  ☐ Spot-check results for reasonableness")

print("\n☐ STEP 5: DOCUMENT & COMMUNICATE")
print("  ☐ Add comments explaining each decision")
print("  ☐ Create summary of rows dropped / values filled")
print("  ☐ Flag any imputed values for later reference")
print("  ☐ Include before/after statistics")

print("\n" + "="*70)
print("EXAMPLE: Complete Workflow")
print("="*70)

print("\n# Step 1: Detect")
print("print(employees_df.info())  # Shows missing counts")
print("print(employees_df.isna().sum())  # Detailed by column")

print("\n# Step 2: Plan")
print("# Decision: Drop rows missing Salary/Department (critical)")
print("#           Drop PhoneNumber column (too many missing)")
print("#           Fill HireDate with 'Unknown'")

print("\n# Step 3: Implement")
print("employees_clean = employees_df.dropna(subset=['Salary', 'Department'])")
print("employees_clean = employees_clean.drop(columns=['PhoneNumber'])")
print("employees_clean['HireDate'] = employees_clean['HireDate'].fillna('Unknown')")

print("\n# Step 4: Verify")
print("print(f'Before: {employees_df.shape}')")
print("print(f'After: {employees_clean.shape}')")
print("print(employees_clean.info())  # Confirm no missing values")

print("\n# Step 5: Document")
print("# Removed 2 rows missing critical Salary/Department data")
print("# Dropped PhoneNumber column (60% missing)")
print("# Filled HireDate with 'Unknown' for 1 record")

## Summary: Key Takeaways

### Drop Strategies
- **`.dropna()`**: Remove rows with ANY missing values (strictest)
- **`.dropna(subset=[...])`**: Remove rows missing specific columns (flexible)
- **`.dropna(axis=1)`**: Remove entire columns with missing values
- **`.dropna(thresh=n)`**: Remove rows with fewer than n non-null values

### Fill Strategies
- **`.fillna(value)`**: Replace with constant (best for categorical)
- **`.fillna(df.mean())`**: Replace with mean (numeric, normally distributed)
- **`.fillna(df.median())`**: Replace with median (numeric, skewed data)
- **`.fillna(method='ffill')`**: Forward fill (sequential data)
- **`.fillna(method='bfill')`**: Backward fill (sequential data)
- **`.groupby().transform()`**: Fill by group statistics (context-aware)

### When to Drop
- Data is Missing Completely at Random (MCAR)
- Missing data is < 5% of dataset
- Can afford to lose rows
- Non-critical columns

### When to Fill
- Data is Missing at Random (MAR) or Not Missing at Random (MNAR)
- Column is critical to analysis
- Reasonable estimation method exists
- Losing rows would compromise results

### Critical Numeric Columns
- **Risky to drop**: Loses important records, may bias analysis
- **Safer to fill**: Preserves data, uses context for estimation
- **Best practice**: Document imputation, flag imputed values, sensitivity test